#
**ANALISIS DE ENERGÍA EN LAS ZONAS NO INTERCONECTADAS DE COLOMBIA**

##
**OBTENCIÓN DE RUTAS DE DATASETS, BIBLIOTECAS Y LIMPIEZA**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime as dt
import unicodedata

In [2]:
URL1 = "C:/Users/Lenovo/Desktop/ANALISIS ZNI/DATASETS/DIVIPOLA_-_Códigos_cabeceras_-_Centros_poblados_20260126.csv"
URL2 = "C:/Users/Lenovo/Desktop/ANALISIS ZNI/DATASETS/Estado_de_la_prestación_del_servicio_de_energía_en_Zonas_No_Interconectadas_20260122.csv"
URL3 = "C:/Users/Lenovo/Desktop/ANALISIS ZNI/DATASETS/Anexo_ICEE_2023_dpto_mpio.xlsx"


###
**Lectura data localidades**

In [3]:
df_localidades = pd.read_csv(URL1,
                            decimal = ',').drop(2) #se dropea 2 debido a que tiene doble coma.
df_localidades

,Código Departamento,Nombre_departamento,Codigo Municipio,Nombre Municipio,Código Centro Poblado,Nombre Centro Poblado,Tipo*,longitud,Latitud
0,17,CALDAS,17050,ARANZAZU,17050006,SAN RAFAEL,CP,"-75,523505","5,261945"
1,17,CALDAS,17050,ARANZAZU,17050012,LA HONDA,CP,"-75,512562","5,253714"
3,5,ANTIOQUIA,5001,MEDELLÍN,5001001,PALMITAS,CP,"-75,690573","6,343919"
4,5,ANTIOQUIA,5001,MEDELLÍN,5001004,SANTA ELENA,CP,"-75,501293","6,210599"
5,5,ANTIOQUIA,5001,MEDELLÍN,5001009,ALTAVISTA,CP,"-75,643706","6,221429"
...,...,...,...,...,...,...,...,...,...
8156,99,VICHADA,99773,CUMARIBO,99773028,GUACO BAJO,CP,"-70,16446","3,364312"
8157,99,VICHADA,99773,CUMARIBO,99773029,GUACO ALTO,CP,"-70,131371","3,401846"
8158,99,VICHADA,99773,CUMARIBO,99773030,CAÑO BOCÓN,CP,"-68,657583","3,965914"
8159,99,VICHADA,99773,CUMARIBO,99773031,CAMUNIANAE,CP,"-69,059251","4,366269"


####
VERIFICACIÓN DATOS NULOS

In [4]:
df_localidades.isna().sum()

Código Departamento      0
Nombre_departamento      0
Codigo Municipio         0
Nombre Municipio         0
Código Centro Poblado    0
Nombre Centro Poblado    0
Tipo*                    0
longitud                 0
Latitud                  0
dtype: int64

####
PASAR LONGITUD Y LATITUD A FLOAT

In [5]:
df_localidades[['longitud', 'Latitud']] = df_localidades[['longitud', 'Latitud']].replace(",", ".", regex=True).astype(float)

####
PASAR NOMBRES DE COLUMNAS A MAYUSCULAS

In [6]:
df_localidades.columns = df_localidades.columns.str.upper()

####
CAMBIAR NOMBRE DE LA COLUMNA 7 DE TIPO* A TIPO Y RENOMBRAR TODAS LAS COLUMNAS CON NOMBRES SIMILIARES A LAS DF_ZNI

In [7]:
df_localidades.rename(columns = {'TIPO*' : 'TIPO',
                                'CÓDIGO DEPARTAMENTO' : 'ID DEPARTAMENTO',
                                'NOMBRE_DEPARTAMENTO' : 'DEPARTAMENTO',
                                'CODIGO MUNICIPIO' : 'ID MUNICIPIO',
                                'NOMBRE MUNICIPIO' : 'MUNICIPIO',
                                'CÓDIGO CENTRO POBLADO' : 'ID LOCALIDAD',
                                'NOMBRE CENTRO POBLADO' : 'LOCALIDAD'}, 
                      inplace= True)

In [8]:
df_localidades.info()

<class 'pandas.DataFrame'>
Index: 8160 entries, 0 to 8160
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ID DEPARTAMENTO  8160 non-null   int64  
 1   DEPARTAMENTO     8160 non-null   str    
 2   ID MUNICIPIO     8160 non-null   int64  
 3   MUNICIPIO        8160 non-null   str    
 4   ID LOCALIDAD     8160 non-null   int64  
 5   LOCALIDAD        8160 non-null   str    
 6   TIPO             8160 non-null   str    
 7   LONGITUD         8160 non-null   float64
 8   LATITUD          8160 non-null   float64
dtypes: float64(2), int64(3), str(4)
memory usage: 891.8 KB


###
**Lectura data ZNI**

In [9]:
df_ZNI = pd.read_csv(URL2,
                     decimal=',',
                     thousands= '.',
                     parse_dates=["FECHA DE DEMANDA MÁXIMA"],
                     date_format= "%Y %b %d %I:%M:%S %p")
df_ZNI

,ID DEPATAMENTO,DEPARTAMENTO,ID MUNICIPIO,MUNICIPIO,ID LOCALIDAD,LOCALIDAD,AÑO SERVICIO,MES SERVICIO,ENERGÍA ACTIVA,ENERGÍA REACTIVA,POTENCIA MÁXIMA,DÍA DE DEMANDA MÁXIMA,FECHA DE DEMANDA MÁXIMA,PROMEDIO DIARIO EN HORAS
0,91,AMAZONAS,91001,LETICIA,91001000,LETICIA (LETICIA - AMAZONAS),2020,1,3930642,1251191,7768.76,lunes,2020-01-27 14:15:00,24.0000
1,91,AMAZONAS,91540,PUERTO NARIÑO,91540000,PUERTO NARIÑO (PUERTO NARIÑO - AMAZONAS),2020,1,103897,36304,227.04,miércoles,2020-01-22 19:15:00,24.0000
2,91,AMAZONAS,91798,TARAPACÁ (ANM),91798000,TARAPACÁ (TARAPACÁ (ANM) - AMAZONAS),2020,1,22864,9277,88.96,jueves,2020-01-30 19:30:00,10.1900
3,5,ANTIOQUIA,5873,VIGÍA DEL FUERTE,5873001,SAN ANTONIO DE PADUA (VIGÍA DEL FUERTE - ANTIO...,2020,1,5617,1381,53.66,jueves,2020-01-23 19:45:00,4.1300
4,5,ANTIOQUIA,5873,VIGÍA DEL FUERTE,5873002,VEGÁEZ (VIGÍA DEL FUERTE - ANTIOQUIA),2020,1,2217,539,39.07,miércoles,2020-01-29 19:45:00,3.1700
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5983,97,VAUPÉS,97161,CARURÚ,97161000,CARURÚ(CARURÚ - VAUPÉS),2025,11,40640,7582,106.04,viernes,2025-11-07 18:45:00,17.0500
5984,97,VAUPÉS,97666,TARAIRA,97666000,TARAIRA(TARAIRA - VAUPÉS),2025,11,49902,12875,150.92,martes,2025-11-04 12:30:00,14.2833
5985,99,VICHADA,99001,PUERTO CARREÑO,99001000,PUERTO CARREÑO(PUERTO CARREÑO - VICHADA),2025,11,2817887,720573,5522.00,miércoles,2025-11-19 13:30:00,23.9500
5986,99,VICHADA,99001,PUERTO CARREÑO,99001002,CASUARITO(PUERTO CARREÑO - VICHADA),2025,11,14567,7108,85.05,viernes,2025-11-21 16:30:00,7.6250


In [10]:
df_ZNI.info()


<class 'pandas.DataFrame'>
RangeIndex: 5988 entries, 0 to 5987
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   ID DEPATAMENTO            5988 non-null   int64         
 1   DEPARTAMENTO              5988 non-null   str           
 2   ID MUNICIPIO              5988 non-null   int64         
 3   MUNICIPIO                 5988 non-null   str           
 4   ID LOCALIDAD              5988 non-null   int64         
 5   LOCALIDAD                 5988 non-null   str           
 6   AÑO SERVICIO              5988 non-null   int64         
 7   MES SERVICIO              5988 non-null   int64         
 8   ENERGÍA ACTIVA            5988 non-null   int64         
 9   ENERGÍA REACTIVA          5988 non-null   int64         
 10  POTENCIA MÁXIMA           5908 non-null   float64       
 11  DÍA DE DEMANDA MÁXIMA     5907 non-null   str           
 12  FECHA DE DEMANDA MÁXIMA   5908 

####
VERIFICAR VALORES NULOS Y ELIMINAR ESTOS

In [11]:
df_ZNI.isna().sum()

ID DEPATAMENTO               0
DEPARTAMENTO                 0
ID MUNICIPIO                 0
MUNICIPIO                    0
ID LOCALIDAD                 0
LOCALIDAD                    0
AÑO SERVICIO                 0
MES SERVICIO                 0
ENERGÍA ACTIVA               0
ENERGÍA REACTIVA             0
POTENCIA MÁXIMA             80
DÍA DE DEMANDA MÁXIMA       81
FECHA DE DEMANDA MÁXIMA     80
PROMEDIO DIARIO EN HORAS     0
dtype: int64

In [12]:
df_ZNI.dropna(inplace=True)

####
QUITAR TILDES Y PASAR TODOS LOS DÍAS DE LA SEMANA A MAYUSCULA

In [13]:
def eliminar_tildes(texto):
    return unicodedata.normalize('NFD', texto).encode('ascii', 'ignore').decode('utf-8')

In [14]:
df_ZNI['DÍA DE DEMANDA MÁXIMA'] = df_ZNI['DÍA DE DEMANDA MÁXIMA'].apply(lambda x: eliminar_tildes(x).upper())

In [15]:
list(df_ZNI['DÍA DE DEMANDA MÁXIMA'].unique())

['LUNES', 'MIERCOLES', 'JUEVES', 'VIERNES', 'SABADO', 'MARTES', 'DOMINGO']

####
RENOMBRAR COLUMNA DE DEPARTAMENTOS

In [16]:
df_ZNI.rename(columns = {'ID DEPATAMENTO' : 'ID DEPARTAMENTO'}, inplace = True)

###
Lectura Indice de Cobertura de Energía Eléctrica

####
ICEE MUNICIPIOS

In [17]:
df_ICEE_municipios = pd.read_excel(URL3, sheet_name='ICEE_MPIO')
df_ICEE_municipios

,Cod. DANE,Vigencia,Departamento,Municipio,VCS Urbano,VCS Rural,VCS Total,VT Urbano,VT Rural,VT,ICEE Urbano,ICEE Rural,ICEE Total,VSS Urbano,VSS Rural,VSS Total,Excedentes Total
0,5001,2023,Antioquia,Medellín,1002117,20700,1022817,1002117,20700,1022817,1.000000,1.000000,1.000000,0,0,0,25852
1,5002,2023,Antioquia,Abejorral,3977,4870,8847,4311,5984,10295,0.922524,0.813837,0.859349,334,1114,1448,Sin excedentes
2,5004,2023,Antioquia,Abriaquí,416,671,1087,567,782,1349,0.733686,0.858056,0.805782,151,111,262,Sin excedentes
3,5021,2023,Antioquia,Alejandría,1237,1096,2333,1533,1277,2810,0.806915,0.858262,0.830249,296,181,477,Sin excedentes
4,5030,2023,Antioquia,Amagá,6441,6195,12636,6633,7047,13680,0.971054,0.879097,0.923684,192,852,1044,Sin excedentes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1117,97889,2023,Vaupés,Yavaraté,0,0,0,0,186,186,0.000000,0.000000,0.000000,0,186,186,Sin excedentes
1118,99001,2023,Vichada,Puerto Carreño,5903,1695,7598,5903,1958,7861,1.000000,0.865679,0.966544,0,263,263,Sin excedentes
1119,99524,2023,Vichada,La Primavera,2371,1433,3804,2371,1433,3804,1.000000,1.000000,1.000000,0,0,0,502
1120,99624,2023,Vichada,Santa Rosalía,897,73,970,897,532,1429,1.000000,0.137218,0.678796,0,459,459,Sin excedentes


##### 
Veririficación valores nulos

In [18]:
df_ICEE_municipios.isna().sum()

Cod. DANE           0
Vigencia            0
Departamento        0
Municipio           0
VCS Urbano          0
VCS Rural           0
VCS Total           0
VT Urbano           0
VT Rural            0
VT                  0
ICEE Urbano         0
ICEE Rural          0
ICEE Total          0
VSS Urbano          0
VSS Rural           0
VSS Total           0
Excedentes Total    0
dtype: int64

#####
Cambio de todos los atributos a mayúscula e igualar algunos al nombre del df_ZNI. 

In [19]:
df_ICEE_municipios.columns = df_ICEE_municipios.columns.str.upper()

In [20]:
df_ICEE_municipios.rename(columns = {'COD. DANE' : 'ID MUNICIPIO'}, 
                          inplace = True)

#####
Drop columna vigencia y datos innecesarios para el analisis.

In [21]:
#Se dropea columna vigencia ya que todos los datos son del 2023
df_ICEE_municipios.drop(columns = 'VIGENCIA', 
                        inplace = True)

In [22]:
"""Esta condición nos permite mirar si se pueden eliminar todos los valores donde en la fila de excedentes total es = a sin exedentes,
a su vez verificar si no hay ninguna vivienda rural con exedentes"""

df_ICEE_municipios[(df_ICEE_municipios['EXCEDENTES TOTAL'] != 'Sin excedentes') & (df_ICEE_municipios['VSS RURAL'] > 0)]



,ID MUNICIPIO,DEPARTAMENTO,MUNICIPIO,VCS URBANO,VCS RURAL,VCS TOTAL,VT URBANO,VT RURAL,VT,ICEE URBANO,ICEE RURAL,ICEE TOTAL,VSS URBANO,VSS RURAL,VSS TOTAL,EXCEDENTES TOTAL


In [23]:
#SE dropea las columnas diferentes a sin excedentes

"""indices_con_excedentes = df_ICEE_municipios[df_ICEE_municipios['EXCEDENTES TOTAL'] != 'Sin excedentes'].index
df_ICEE_municipios.drop(indices_con_excedentes, inplace = True)"""

"indices_con_excedentes = df_ICEE_municipios[df_ICEE_municipios['EXCEDENTES TOTAL'] != 'Sin excedentes'].index\ndf_ICEE_municipios.drop(indices_con_excedentes, inplace = True)"

In [24]:
"""df_ICEE_municipios.reset_index(inplace = True, drop= True)"""

'df_ICEE_municipios.reset_index(inplace = True, drop= True)'

In [25]:
"""df_ICEE_municipios.sort_values(by = 'VSS RURAL', ascending= False)"""

"df_ICEE_municipios.sort_values(by = 'VSS RURAL', ascending= False)"

In [26]:
"""df_ICEE_municipios.drop(columns = 'EXCEDENTES TOTAL', inplace = True)"""

"df_ICEE_municipios.drop(columns = 'EXCEDENTES TOTAL', inplace = True)"

In [27]:
df_ICEE_municipios.info()

<class 'pandas.DataFrame'>
RangeIndex: 1122 entries, 0 to 1121
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID MUNICIPIO      1122 non-null   int64  
 1   DEPARTAMENTO      1122 non-null   str    
 2   MUNICIPIO         1122 non-null   str    
 3   VCS URBANO        1122 non-null   int64  
 4   VCS RURAL         1122 non-null   int64  
 5   VCS TOTAL         1122 non-null   int64  
 6   VT URBANO         1122 non-null   int64  
 7   VT RURAL          1122 non-null   int64  
 8   VT                1122 non-null   int64  
 9   ICEE URBANO       1122 non-null   float64
 10  ICEE RURAL        1122 non-null   float64
 11  ICEE TOTAL        1122 non-null   float64
 12  VSS URBANO        1122 non-null   int64  
 13  VSS RURAL         1122 non-null   int64  
 14  VSS TOTAL         1122 non-null   int64  
 15  EXCEDENTES TOTAL  1122 non-null   object 
dtypes: float64(3), int64(10), object(1), str(2)
memory us

##
TRANSFORMACIÓN DE DATOS

###
Unión de divipola en df_ZNI

In [28]:
df_municipios = df_localidades.groupby(['ID DEPARTAMENTO', 'ID MUNICIPIO'], as_index=False).first()[
    ['ID DEPARTAMENTO', 'ID MUNICIPIO', 'DEPARTAMENTO', 'MUNICIPIO', 'LONGITUD', 'LATITUD']]

In [29]:
df_ZNI_limpio = pd.merge(df_ZNI,
                          df_municipios,
                          how= 'left',
                          on= ['ID DEPARTAMENTO', 'ID MUNICIPIO'],
                          validate='many_to_one')

In [30]:
df_ZNI_limpio.drop(columns = ['DEPARTAMENTO_x', 'MUNICIPIO_x'], inplace = True)

DEPARTAMENTOS = df_ZNI_limpio.pop("DEPARTAMENTO_y")
MUNICIPIOS = df_ZNI_limpio.pop("MUNICIPIO_y")

df_ZNI_limpio.insert(1, 'DEPARTAMENTO', DEPARTAMENTOS) 
df_ZNI_limpio.insert(3, 'MUNICIPIO', MUNICIPIOS)


In [31]:
df_ZNI_limpio['LOCALIDAD'] = df_ZNI_limpio['LOCALIDAD'].str.split("(").str[0].str.strip()

df_ZNI_limpio['LOCALIDAD'] = df_ZNI_limpio['LOCALIDAD'].apply(eliminar_tildes)

In [44]:
df_promedio_horas_servicio = pd.DataFrame(df_ZNI_limpio.groupby(df_ZNI_limpio['FECHA DE DEMANDA MÁXIMA'].dt.strftime('%Y'))['PROMEDIO DIARIO EN HORAS'].mean())
df_promedio_horas_servicio

,PROMEDIO DIARIO EN HORAS
FECHA DE DEMANDA MÁXIMA,
2020,11.336854
2021,12.210488
2022,12.799361
2023,12.352222
2024,11.634686
2025,11.158397


In [46]:
df_promedio_horas_servicio.var()

PROMEDIO DIARIO EN HORAS    0.409169
dtype: float64